# 10. Audit Trail

Every Arc deployment — personal, enterprise, or federal — has the same four
non-negotiable pillars: **identity**, **sign**, **authorize**, **audit**.
This notebook covers the last one. We trace how an `arcllm` call produces a
structured audit event, how that event is routed through `arctrust.audit.emit`,
and how pluggable sinks make the same emission compatible with a durable
signed hash-chained file, or a custom destination of your choosing.

You will learn:

- Why `arcllm.modules.AuditModule` is now a *thin emitter* and what it actually does.
- The `AuditEvent` schema and the `AuditSink` Protocol.
- How `WormSink` — the single durable, append-only, Ed25519-signed hash-chained
  audit log — replaced the old unchained `JsonlSink` + in-memory-only
  `SignedChainSink` split with one restart-safe, tamper-evident sink.
- How to write a custom sink in five lines and bolt it on.
- How `load_model(audit=...)` threads sinks through the stack.

Hash-chained `TraceStore` deep-dive — RFC 8785 JCS canonical JSON, daily
rotation, tamper detection — lives in [walkthroughs/arcllm/14-trace-store.ipynb](14-trace-store.ipynb).
The arctrust-side audit primer (sink internals, NullSink, signed-chain verification)
lives in [walkthroughs/arctrust/04-audit-sinks.ipynb](../arctrust/04-audit-sinks.ipynb).


## 1. Setup

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

Mock-first walkthrough. We will fabricate `LLMResponse` values and use a
test adapter so every cell runs without an API key. Where files are written
we use `tempfile.TemporaryDirectory` so nothing leaks out of the notebook.

In [ ]:
import json
import logging
import tempfile
from pathlib import Path
from unittest.mock import AsyncMock, MagicMock

from arctrust.audit import (
    AuditEvent,
    AuditSink,
    NullSink,
    WormSink,
    emit,
    verify_chain,
)
from arctrust.keypair import generate_keypair
from arctrust.signer import ED25519, InProcessSigner

from arcllm.modules.audit import AuditModule
from arcllm.types import LLMProvider, LLMResponse, Message, Tool, ToolCall, Usage

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
print("audit primitives loaded")

## 2. Audit philosophy in Arc

Three properties drive the design.

**Single point of emit.** Every security-relevant action — tool dispatch,
policy decision, LLM call — flows through one call site: `arctrust.audit.emit`.
There is no parallel audit pipeline per package. Sinks fan out from that one
function.

**Pluggable sinks.** The shape of the destination is the caller's problem.
`AuditSink` is a `Protocol` with one method, `write(event)`. A durable signed
chain, a UI bridge, a NATS topic, a syslog forwarder, an in-memory test
capture — all the same interface.

**Auditing never breaks the caller.** NIST 800-53 AU-5 says the audit
subsystem must not interrupt the operation being audited. `emit()` enforces
this by catching every exception from a sink, logging at `WARNING`, and
returning normally.

Tier (`personal` / `enterprise` / `federal`) is **stringency metadata**, not
a gate. Every tier runs the same `WormSink` — the single durable, append-only,
Ed25519-signed hash-chained audit log (it replaced an earlier split between an
unchained `JsonlSink` and an in-memory-only `SignedChainSink`). All still call
the same `emit()`.

In [ ]:
# A minimal AuditEvent — the universal audit shape across Arc.
evt = AuditEvent(
    actor_did="did:arc:agent:demo-007",
    action="llm.invoke",
    target="anthropic/claude-sonnet-4-6",
    outcome="allow",
    tier="personal",
    request_id="req-001",
)
print(evt.action, evt.outcome, evt.target)
print("ts (auto-populated):", evt.ts)

Two things to notice:

- `ts` was filled in by a Pydantic `model_validator` — you never have to
  remember to pass it.
- The model is **frozen** (`model_config = ConfigDict(frozen=True)`), so an
  event cannot be mutated after creation. Tampering must happen at the sink
  layer, where signed chains catch it.

In [ ]:
# Frozen — assignment raises at runtime.
try:
    evt.outcome = "deny"  # type: ignore[misc]
except Exception as e:
    print(type(e).__name__, "-", e)

## 3. `AuditModule` — the thin emitter

In `arcllm` v0.4.0 the audit module was deliberately reduced. It used to
carry its own log format, its own structured-log helper, and its own PII
policy. All of that moved either upstream (to `arctrust.audit`) or downstream
(to `TraceStore`, covered in notebook 14).

What `AuditModule` does today, in full:

1. Wraps `invoke()` and forwards the call to the inner provider.
2. Computes a few audit metadata fields (message count, content length,
   tool counts).
3. Annotates an OTel span (`arcllm.audit`) with those attributes.
4. Emits a structured log line at `INFO` for compliance pipelines that
   scrape stdout.
5. Optional: at `DEBUG`, with a double opt-in, logs sanitized message
   and response bodies.

Crucially, every event AuditModule produces is *also* available to
`arctrust.audit.emit` if you have wired a sink — that integration happens
at the registry layer (covered in §8).

In [ ]:
# A reusable mock inner adapter for the rest of the notebook.
def make_inner(
    *,
    name: str = "anthropic",
    response: LLMResponse | None = None,
    raises: Exception | None = None,
) -> LLMProvider:
    inner = MagicMock(spec=LLMProvider)
    inner.name = name
    inner.model_name = "claude-sonnet-4-6"
    if raises is not None:
        inner.invoke = AsyncMock(side_effect=raises)
    else:
        inner.invoke = AsyncMock(
            return_value=response
            or LLMResponse(
                content="Hello there!",
                usage=Usage(input_tokens=100, output_tokens=50, total_tokens=150),
                model="claude-sonnet-4-6",
                stop_reason="end_turn",
            )
        )
    return inner


module = AuditModule({}, make_inner())
messages = [
    Message(role="system", content="You are a helpful assistant."),
    Message(role="user", content="What is 2+2?"),
]
resp = await module.invoke(messages)
print("response:", resp.content)

The log line it produced is what compliance pipelines parse: `provider`,
`model`, `message_count`, `stop_reason`, `content_length`, plus optional
`tools_provided` and `tool_calls`. No PII. No raw content. That is the
default mode and the right mode for production.

In [ ]:
# Tool-call response — conditional fields appear only when relevant.
tool_resp = LLMResponse(
    content=None,
    tool_calls=[
        ToolCall(id="call_1", name="search", arguments={"query": "arc"}),
        ToolCall(id="call_2", name="calc", arguments={"expr": "2+2"}),
    ],
    usage=Usage(input_tokens=20, output_tokens=10, total_tokens=30),
    model="claude-sonnet-4-6",
    stop_reason="tool_use",
)
tools = [Tool(name="search", description="Search the web", parameters={"type": "object"})]

tool_module = AuditModule({}, make_inner(response=tool_resp))
_ = await tool_module.invoke(messages, tools=tools)

Two helpful invariants in `AuditModule`:

- It **never swallows exceptions** from the inner provider. If the LLM
  call raises, the agent sees it.
- It **rejects unknown config keys** at construction time, so typos
  surface immediately instead of silently doing nothing.

In [ ]:
from arcllm.exceptions import ArcLLMConfigError

# Exception propagation: the inner failure is not eaten.
fail_module = AuditModule({}, make_inner(raises=RuntimeError("upstream 503")))
try:
    await fail_module.invoke(messages)
except RuntimeError as e:
    print("propagated:", e)

# Strict config-key validation.
try:
    AuditModule({"include_mesages": True}, make_inner())  # typo
except ArcLLMConfigError as e:
    print("caught typo:", e)

## 4. `arctrust` sinks — the architecture

`arctrust.audit` exposes these pieces:

| Symbol | Role |
|---|---|
| `AuditEvent` | Frozen Pydantic model — the universal event shape. |
| `AuditSink` | `Protocol` with `write(event) -> None`. |
| `WormSink` | The single durable, append-only, Ed25519-signed hash-chained audit log — the compliance system of record. Restart-safe, single-writer (flock), self-rotating. |
| `NullSink` | Drops everything. Used by tests. |
| `emit(event, sink)` | Safe dispatch — swallows sink errors per AU-5. |
| `verify_chain(path, public_key)` | Lock-free read-path verification of a `WormSink` chain on disk. |

Because `AuditSink` is a `runtime_checkable` Protocol, **anything** with
the right method signature works. No inheritance, no registration.

In [ ]:
# Quick sanity check of the protocol shape.
print("AuditSink methods:", [m for m in dir(AuditSink) if not m.startswith("_")])

# NullSink is the fallback when audit is intentionally disabled.
null = NullSink()
emit(evt, null)
print("NullSink records:", null.records)

`emit()` is the **only** call site you need to remember. It runs the sink
in a try/except and never re-raises. Even if a sink crashes — disk full,
network unreachable, third-party bug — the call site keeps moving.

In [ ]:
class CrashSink:
    """A sink that always raises — to prove emit() catches it."""

    def write(self, event: AuditEvent) -> None:
        raise RuntimeError("disk on fire")


# emit() must not raise even when the sink does.
emit(evt, CrashSink())
print("survived a crashing sink")

## 5. `WormSink` — construction and durable writes

`WormSink` is the single durable, append-only, Ed25519-signed hash-chained
audit log — the compliance system of record. It replaced an earlier split
between an unchained `JsonlSink` (durable but not tamper-evident) and an
in-memory-only `SignedChainSink` (tamper-evident but lost on restart).
`WormSink` is both: every `write()` appends one JSON line
`{seq, event, prev_hash, event_hash, algorithm, signature}` to an append-only
`0o600` file, the chain is signed with an `arctrust.signer.Signer` (Ed25519 or
ECDSA-P256), and the chain tip/seq are recovered from the file tail on
construction — so it survives process restarts. It also holds an exclusive
`flock` for its lifetime (single-writer) and self-rotates at `max_records`
/ `max_bytes`.

Construction takes a `Path` and a `Signer` — not a bare env-var key like
`arcllm`'s request signer. Here we mint a fresh Ed25519 keypair with
`generate_keypair()` and wrap it in an `InProcessSigner`.

In [ ]:
chain_dir = Path(tempfile.mkdtemp(prefix="arc-nb10-"))
chain_path = chain_dir / "audit.worm"

operator = generate_keypair()
signer = InProcessSigner(operator.private_key, ED25519)
chain = WormSink(chain_path, signer)

# Emit a few events.
for action, outcome in [
    ("llm.invoke", "allow"),
    ("tool.call", "allow"),
    ("policy.evaluate", "deny"),
]:
    e = AuditEvent(
        actor_did="did:arc:agent:demo-007",
        action=action,
        target="anthropic/claude-sonnet-4-6",
        outcome=outcome,
        tier="enterprise",
    )
    emit(e, chain)

print("file size:", chain_path.stat().st_size, "bytes")
print("chain tip:", chain.chain_tip[:32], "...")

Now read it back. Each line is parseable as a single JSON document carrying
the chain fields (`seq`, `event`, `prev_hash`, `event_hash`, `algorithm`,
`signature`) alongside the `AuditEvent` payload. Production pipelines
stream-parse this file — there is no all-or-nothing deserialization risk.

In [ ]:
for raw in chain_path.read_text().splitlines():
    rec = json.loads(raw)
    print(
        f"seq={rec['seq']} {rec['event']['action']:16s} -> {rec['event']['outcome']:6s} "
        f"| algo={rec['algorithm']} | event_hash={rec['event_hash'][:16]}..."
    )

In [ ]:
# WormSink takes a real Path — not a file-like object like the old JsonlSink did
# (the flock single-writer invariant needs a real file descriptor). It sets the
# 0o600 perm on open, automatically — no manual chmod() required.
import stat

mode = stat.S_IMODE(chain_path.stat().st_mode)
print(f"file perms: {oct(mode)}")
assert mode == 0o600, "WormSink must open its file at 0o600"

`WormSink` only hardens the *file* to `0o600` — unlike arcllm's own
`TraceStore` (notebook 14 and `12-security-layer.ipynb` §6), it does **not**
chmod the parent directory to `0o700`. If you're deploying a `WormSink`
directly, harden the containing directory yourself as part of provisioning.

In [ ]:
# Manual directory hardening — WormSink does not do this for you.
chain_dir.chmod(0o700)
print(f"file perms: 0o{chain_path.stat().st_mode & 0o777:o}")
print(f"dir perms:  0o{chain_dir.stat().st_mode & 0o777:o}")

## 6. Verifying the chain — tamper detection

Each record's `event_hash` commits `(seq, prev_hash, event)`, and that hash
is Ed25519/ECDSA-signed. `chain.verify_chain()` (equivalently the module-level
`verify_chain(path, public_key)`, which needs **no write lock** — it streams
the file read-only) recomputes every hash, checks the signature, confirms
`seq` is contiguous from 0, and confirms the chain links to the genesis
anchor. Any modification to a stored record breaks verification.

This is the file-backed primitive `arcllm`'s own `TraceStore` (notebook 14)
builds on for hash-chained, RFC-8785-canonical-JSON per-call records.
Verification semantics for the standalone primitive — a separate verifier
key, `read_verified_anchor()`, the AU-9/AU-10/AU-11 mapping — are covered in
[walkthroughs/arctrust/04-audit-sinks.ipynb](../arctrust/04-audit-sinks.ipynb).

In [ ]:
# Reuse the `chain` built in §5 — verify it before touching anything.
print("verify_chain() (instance method):", chain.verify_chain())
print(
    "verify_chain() (module-level, lock-free read path):",
    verify_chain(chain_path, operator.public_key),
)

Every record carries `seq`, `event`, `prev_hash`, `event_hash`, `algorithm`,
and `signature`. `verify_chain()` recomputes each `event_hash` from
`(seq, prev_hash, event)`, checks the signature, and confirms `seq`/`prev_hash`
form an unbroken chain — a single mutated byte anywhere causes it to return
`False`.

`WormSink` holds an exclusive `flock` for its lifetime (the single-writer
invariant), so we release it with `close()` before corrupting the file
out-of-band — the same thing an operator inspecting a chain after the
writing process exits would do.

In [ ]:
# Tamper demo: mutate the middle record's outcome directly on disk.
chain.close()  # release the single-writer flock first

lines = chain_path.read_text().splitlines()
record = json.loads(lines[1])
record["event"]["outcome"] = "deny"
lines[1] = json.dumps(record)
chain_path.write_text("\n".join(lines) + "\n")

print("verify_chain() after tamper:", verify_chain(chain_path, operator.public_key))

For full coverage of `WormSink` internals — rotation, crash recovery of a
torn write, `read_verified_anchor()`, and the operator-key handling story —
see arctrust's audit-sinks notebook. The point here is that **the wiring
from `arcllm` is identical to the wiring for any other sink** — you only
swap the constructor.

## 7. Custom sinks — five lines

Any class with `write(event: AuditEvent) -> None` satisfies the protocol.
We will build two: an in-memory capture sink for tests, and a metrics sink
that aggregates outcomes.

In [ ]:
class CaptureSink:
    """Append every event to an in-memory list — for tests."""

    def __init__(self) -> None:
        self.events: list[AuditEvent] = []

    def write(self, event: AuditEvent) -> None:
        self.events.append(event)


capture = CaptureSink()
for outcome in ("allow", "allow", "deny", "allow"):
    emit(
        AuditEvent(
            actor_did="did:arc:agent:demo-007",
            action="tool.call",
            target="shell.exec",
            outcome=outcome,
        ),
        capture,
    )
print("captured:", len(capture.events))
print("outcomes:", [e.outcome for e in capture.events])

Type-checkers verify Protocol conformance without any inheritance —
`CaptureSink` is recognized as an `AuditSink` because it has the right
method, period.

In [ ]:
# isinstance() works because AuditSink is @runtime_checkable.
print("is AuditSink:", isinstance(capture, AuditSink))

In [ ]:
from collections import Counter


class MetricsSink:
    """Aggregate outcomes by action — useful as a Prometheus shim."""

    def __init__(self) -> None:
        self.counts: Counter[tuple[str, str]] = Counter()

    def write(self, event: AuditEvent) -> None:
        self.counts[(event.action, event.outcome)] += 1


metrics = MetricsSink()
for outcome in ("allow", "allow", "deny"):
    emit(
        AuditEvent(
            actor_did="did:arc:agent:demo-007",
            action="llm.invoke",
            target="anthropic/claude-sonnet-4-6",
            outcome=outcome,
        ),
        metrics,
    )
for key, n in metrics.counts.items():
    print(f"  {key[0]:>20} {key[1]:>5}: {n}")

### Fan-out: many sinks, one event

There is no built-in `MultiSink` — you don't need one. Wrap your
destinations in a tuple and emit to each. Each `emit()` call is
independently exception-safe, so a failure in one sink does not affect
the others.

In [ ]:
class FanoutSink:
    """Write the same event to every wrapped sink, individually safe."""

    def __init__(self, *sinks: AuditSink) -> None:
        self._sinks = sinks

    def write(self, event: AuditEvent) -> None:
        for s in self._sinks:
            emit(event, s)  # each is wrapped — one bad sink doesn't poison others


# `chain` was closed in §6's tamper demo — open a fresh WormSink for this demo
# rather than fan into a sink whose file we just hand-corrupted.
fanout_dir = Path(tempfile.mkdtemp(prefix="arc-nb10-fanout-"))
fanout_chain = WormSink(fanout_dir / "audit.worm", InProcessSigner(operator.private_key, ED25519))

fanout = FanoutSink(fanout_chain, capture, metrics, CrashSink())
emit(
    AuditEvent(
        actor_did="did:arc:agent:demo-007",
        action="llm.invoke",
        target="anthropic/claude-sonnet-4-6",
        outcome="allow",
    ),
    fanout,
)
print("fanout chain records now:", len((fanout_dir / "audit.worm").read_text().splitlines()))
print("capture events now:", len(capture.events))

## 8. Wiring `AuditModule` into `load_model`

From the agent author's perspective the audit module is just a kwarg.
`audit=True` enables it with config defaults, `audit=False` turns it off,
and `audit={"include_messages": True}` switches on the (DEBUG-gated)
content logging for dev environments.

The stack order is fixed:

```
Otel → Queue → Telemetry → CircuitBreaker → Audit → Security →
Retry → Fallback → RateLimit → [Router|Adapter]
```

Audit sits **above** Retry and Fallback on purpose — it logs the *final*
result the agent receives, not every intermediate retry attempt. That
keeps the audit volume sane and the meaning clear: one audit line equals
one logical agent-visible call.

In [ ]:
from arcllm.registry import clear_cache, load_model

# Set placeholder credentials so registry construction succeeds even though
# we will not actually hit the network in this notebook.
os.environ.setdefault("ANTHROPIC_API_KEY", "test-key")
clear_cache()

model = load_model("anthropic", audit=True)


def stack_layers(m: object) -> list[str]:
    layers, cur = [], m
    while hasattr(cur, "_inner"):
        layers.append(type(cur).__name__)
        cur = cur._inner  # type: ignore[attr-defined]
    layers.append(type(cur).__name__)
    return layers


print("layers:", " -> ".join(stack_layers(model)))

In [ ]:
# Same call, full stack — observe where AuditModule sits.
clear_cache()
full = load_model(
    "anthropic",
    telemetry=True,
    audit=True,
    retry=True,
    rate_limit=True,
)
print("full stack:", " -> ".join(stack_layers(full)))

### Threading sinks through to a real run

For a complete federal-grade pipeline you compose three things:

1. **arcllm `AuditModule`** for per-call structured logs (this notebook).
2. **`TraceStore`** for hash-chained, durable per-call records (notebook 14).
3. **`arctrust.audit.emit` + sinks** for cross-package events like policy
   decisions and tool dispatches.

All three coexist. The pattern below sketches what an enterprise tier
agent's wiring looks like — entirely mock so you can step through it.

In [ ]:
# Mock-only sketch: combine a CaptureSink (for tests/UI bridge) with
# AuditModule's per-call log line. We bypass the real httpx adapter by
# wrapping our mock inner directly.
shared_capture = CaptureSink()

audit_mod = AuditModule({}, make_inner())
_ = await audit_mod.invoke(messages)

# Independently, emit a higher-level audit event for the same call.
emit(
    AuditEvent(
        actor_did="did:arc:agent:demo-007",
        action="llm.invoke",
        target="anthropic/claude-sonnet-4-6",
        outcome="allow",
        tier="enterprise",
        request_id="req-101",
    ),
    shared_capture,
)
print("capture saw:", shared_capture.events[-1].action, "->", shared_capture.events[-1].outcome)

Why two emissions? Because the two layers carry different information:

- `AuditModule`'s log line is **operational** — provider/model/duration/tokens,
  consumed by SIEMs and dashboards.
- The `arctrust.audit` event is **forensic** — actor DID, target, outcome,
  classification, tier — consumed by compliance auditors and incident
  response.

In a real agent the second emission is wired by the orchestrator (`arcagent`)
around tool dispatch and policy evaluation, not inside `arcllm`. The
`arctrust` notebooks cover that side.

## 9. Going deeper

**[walkthroughs/arctrust/04-audit-sinks.ipynb](../arctrust/04-audit-sinks.ipynb)**
— the arctrust-side primer. Full coverage of `AuditEvent` fields
(`classification`, `tier`, `request_id`, `payload_hash`), `NullSink`
semantics, signed-chain verification with a separate verifier key, and
the AU-5 / AU-9 / AU-11 mapping.

**[walkthroughs/arcllm/14-trace-store.ipynb](14-trace-store.ipynb)** —
the durable, file-backed, hash-chained `TraceStore`. Explains RFC 8785
JCS canonical JSON, daily rotation, cursor-based pagination,
`verify_chain()` on warm start, and `TraceRecord` integration with the
telemetry callback chain.

**[walkthroughs/arcllm/09-telemetry-module.ipynb](09-telemetry-module.ipynb)**
— sister module. Same shape (a `BaseModule`-derived wrapper around
`invoke()`), different concern (timings + tokens + cost vs. compliance
metadata).

**[walkthroughs/arcllm/12-security-layer.ipynb](12-security-layer.ipynb)**
— combines audit with input/output redaction so PII never reaches the
audit pipeline in the first place.

## 10. Summary

- The Four Pillars are universal — every Arc tier audits.
- `arctrust.audit.emit` is the single point of audit emission across all
  packages. Sinks are pluggable via the `AuditSink` Protocol.
- `AuditEvent` is a frozen Pydantic model with auto-populated timestamps.
- `WormSink` is the single durable, append-only, Ed25519-signed hash-chained
  audit log — it replaced the old unchained `JsonlSink` + in-memory-only
  `SignedChainSink` split. Restart-safe (recovers chain tip from the file
  tail), single-writer (`flock`), self-rotating, `0o600` file perms out of
  the box (directory hardening is still the caller's job).
- `verify_chain()` (instance method or the lock-free module-level function)
  fails on any tamper — mutated bytes, forged signatures, or sequence gaps.
- A custom sink is one method (`write`). Fan-out is a tuple of `emit()`
  calls, each independently exception-safe.
- `arcllm.modules.AuditModule` is a thin emitter — it wraps `invoke()`,
  annotates an OTel span, logs one structured line per call, and forwards
  exceptions. Old responsibilities (hash chain, RFC 8785 JCS, file rotation)
  live in `TraceStore` / `WormSink`.
- `load_model(audit=...)` slots `AuditModule` between `CircuitBreaker`
  and `Security` so audit logs the *final* agent-visible result.

**Public API exercised in this notebook**

| Symbol | Where |
|---|---|
| `arcllm.modules.AuditModule` | §3, §8 |
| `arcllm.load_model(audit=...)` | §8 |
| `arctrust.audit.AuditEvent` | §2, §4–§7 |
| `arctrust.audit.AuditSink` | §4, §7 |
| `arctrust.audit.WormSink` | §5, §6, §7 |
| `arctrust.audit.verify_chain` | §6 |
| `arctrust.audit.NullSink` | §4 |
| `arctrust.audit.emit` | §2–§8 |
| `arctrust.signer.InProcessSigner`, `arctrust.keypair.generate_keypair` | §5 |

**Next:** [11-otel-export.ipynb](11-otel-export.ipynb) — exporting these
spans (and friends from Telemetry) over OpenTelemetry.